# GEMM 教学版

这个 notebook 用最小可验证的方式讲清楚 GEMM 的 baseline：定义计算、编译到 LLVM、执行、验证结果。

## 运行口径

- 当前环境优先跑 CPU baseline
- GPU 版本需要 CUDA 版 TVM 和 NVIDIA 驱动
- `.py` 版本保留完整演进，这个 notebook 负责教学展示

In [ ]:
import time
import numpy as np
import tvm
from tvm import te
from tvm.ir import IRModule
from tvm.runtime._tensor import tensor as tvm_tensor

In [ ]:
M = N = K = 256
A = te.placeholder((M, K), name="A", dtype="float32")
B = te.placeholder((K, N), name="B", dtype="float32")
k = te.reduce_axis((0, K), name="k")
C = te.compute((M, N), lambda i, j: te.sum(A[i, k] * B[k, j], axis=k), name="C")

prim_func = te.create_prim_func([A, B, C])
mod = IRModule({"matmul_cpu": prim_func})
f = tvm.build(mod, target="llvm")
print(f)

In [ ]:
a_np = np.random.randn(M, K).astype("float32")
b_np = np.random.randn(K, N).astype("float32")
c_np = np.zeros((M, N), dtype="float32")

a = tvm_tensor(a_np, device=tvm.cpu(0))
b = tvm_tensor(b_np, device=tvm.cpu(0))
c = tvm_tensor(c_np, device=tvm.cpu(0))

start = time.time()
f(a, b, c)
elapsed_ms = (time.time() - start) * 1000

error = np.max(np.abs(c.numpy() - a_np.dot(b_np)))
print(f"elapsed: {elapsed_ms:.3f} ms")
print(f"max error: {error:.2e}")